<a href="https://colab.research.google.com/github/pedroManuelP/C-digos-em-Python/blob/main/Lista2_do_Projeto_de_Sistemas_de_Controle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [86]:
import numpy as np

def coefAmort(Mp_percent):
    n = np.log(Mp_percent/100)
    d = np.sqrt(np.power(np.pi,2) + np.power(np.log(Mp_percent/100),2))
    return n/d

def freqNatural(coefAmort,tss, tss_percent):
  if(tss_percent == 5):
    wn = 3/(tss*coefAmort)
  elif(tss_percent == 2):
    wn = wn = 4/(tss*coefAmort)
  return wn

def sigmaAngulos(si, arr):
  sigma = 0
  for i in range(len(arr)):
    p = si - arr[i]
    sigma += np.rad2deg(np.atan2(p.imag, p.real))

  return sigma

def phiModulos(si, arr):
  phi = 1
  for i in range(len(arr)):
    p = si - arr[i]
    phi *= np.abs(p)
  return phi

In [91]:
coefGNum = np.array([4, 16]); coefGDen = np.array([1, 4, 4, 0])
g_num = np.polynomial.Polynomial(np.flip(coefGNum))
g_den = np.polynomial.Polynomial(np.flip(coefGDen))
print("G(s) = (" + str(g_num) + ")/"); print("(" + str(g_den) + ")")

coefHNum = np.array([1]); coefHDen = np.array([1])
h_num = np.polynomial.Polynomial(np.flip(coefHNum))
h_den = np.polynomial.Polynomial(np.flip(coefHDen))
print("\nH(s) = (" + str(h_num) + ")/"); print("(" + str(h_den) + ")")

G(s) = (16.0 + 4.0·x)/
(0.0 + 4.0·x + 4.0·x² + 1.0·x³)

H(s) = (1.0)/
(1.0)


In [55]:
Mp_percent = 10; tss = 4; tss_percent = 5

ksi = np.abs(coefAmort(Mp_percent)); ksi = np.round(ksi,4)
wn = freqNatural(ksi,tss,tss_percent); wn = np.round(wn,4)
print("Coeficiente de amortecimento: " + str(ksi))
print("Frequência natural: " + str(wn))

Coeficiente de amortecimento: 0.5912
Frequência natural: 1.2686


In [56]:
wd = wn*np.sqrt(1-np.power(ksi,2)); wd = np.round(wd,4)
print("Frequência amortecida: " + str(wd))

s1 = np.round(complex(-ksi*wn,wd), 4); s2 = np.round(complex(-ksi*wn,-wd), 4)

Frequência amortecida: 1.0232


In [57]:
print("s1: " + str(s1))
print("s2: " + str(s2))

s1: (-0.75+1.0232j)
s2: (-0.75-1.0232j)


In [94]:
# Controlador PD

polos = (g_den*h_den).roots(); zeros = (g_num*h_num).roots()  # polos e zeros da FT de malha fechada do sistema
phiZ = sigmaAngulos(s1, polos) - sigmaAngulos(s1, zeros) - 180
z = -s1.real + s1.imag/np.tan(np.deg2rad(phiZ))
zeros = np.append(zeros, -z); print("Zero(s) do controlador projetado: " + str(round(z, 4)))

kt = phiModulos(s1, polos)/phiModulos(s1, zeros)
kc = kt/(coefGNum[0]*coefHNum[0])
print("Ganho do controlador projetado: " + str(round(kc, 4)))

kd = kc; kp = kd*z
print("\nGanho proporcional do controlador: " + str(round(kp, 4)))
print("Ganho derivativo do controlador: " + str(round(kd, 4)))

Zero(s) do controlador projetado: 8.6602
Ganho do controlador projetado: 0.0305

Ganho proporcional do controlador: 0.2637
Ganho derivativo do controlador: 0.0305


In [38]:
# Controlador PI

polos = (g_den*h_den).roots(); zeros = (g_num*h_num).roots()  # polos e zeros da FT de malha fechada do sistema
polos = np.append(polos, 0)

phiZ = sigmaAngulos(s1, polos) - sigmaAngulos(s1, zeros) - 180
z = -s1.real + s1.imag/np.tan(np.deg2rad(phiZ))
zeros = np.append(zeros, -z); print("Zero(s) do controlador projetado: " + str(round(z, 4)))

kt = phiModulos(s1, polos)/phiModulos(s1, zeros)
kc = kt/(coefGNum[0]*coefHNum[0])
print("Ganho do controlador projetado: " + str(round(kc, 4)))

kp = kc; ki = kp*z
print("\nGanho proporcional do controlador: " + str(round(kp, 4)))
print("Ganho integrativo do controlador: " + str(round(ki, 4)))

Zero(s) do controlador projetado: 3.4286
Ganho do controlador projetado: 7.0

Ganho proporcional do controlador: 7.0
Ganho integrativo do controlador: 24.0


In [53]:
# Controlador PID

polos = (g_den*h_den).roots(); zeros = (g_num*h_num).roots()  # polos e zeros da FT de malha fechada do sistema
polos = np.append(polos, 0)

phiZ = (sigmaAngulos(s1, polos) - sigmaAngulos(s1, zeros) - 180)/2
z = -s1.real + s1.imag/np.tan(np.deg2rad(phiZ))
zeros = np.append(zeros, -z); print("Zero(s) do controlador projetado: " + str(round(z, 4)))

kt = phiModulos(s1, polos)/phiModulos(s1, zeros)
kc = kt/(coefGNum[0]*coefHNum[0])
print("Ganho do controlador projetado: " + str(round(kc, 4)))

kd = kc; kp = 2*z*kd; ki = kd*np.power(z,2);
print("\nGanho proporcional do controlador: " + str(round(kp, 4)))
print("Ganho integrativo do controlador: " + str(round(ki, 4)))
print("Ganho derivativo do controlador: " + str(round(kd, 4)))

Zero(s) do controlador projetado: 2.049
Ganho do controlador projetado: 3.137

Ganho proporcional do controlador: 12.8558
Ganho integrativo do controlador: 13.1711
Ganho derivativo do controlador: 3.137
